# Deteksi Spam Email/Chat

Notebook ini melatih model deteksi spam menggunakan dataset `data/email.csv` yang sudah disalin ke folder proyek.

Notebook akan mendeteksi otomatis dua format data:

- dataset fitur hitungan kata seperti `data/email.csv` dengan label di kolom `Prediction`
- dataset teks mentah dengan kolom pesan dan label spam/ham

Output utama: akurasi, confusion matrix, classification report, model tersimpan, dan contoh prediksi.

In [1]:
from __future__ import annotations

import csv
import re
from dataclasses import dataclass
from pathlib import Path

import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import FeatureUnion, Pipeline

pd = None
try:
    import pandas as pd
except Exception:
    pd = None

In [12]:
DATASET_PATH = Path(r'c:\MLLECSESSION11\data\email.csv')
MODEL_PATH = Path('spam_model.joblib')

TEXT_COLUMNS = ('text', 'message', 'email', 'content', 'chat')
LABEL_COLUMNS = ('label', 'target', 'class', 'spam', 'prediction', 'category')
IGNORE_COLUMNS = {'email no.', 'prediction', 'label', 'target', 'class', 'spam'}

URL_PATTERN = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
EMAIL_PATTERN = re.compile(r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b', re.IGNORECASE)
WHITESPACE_PATTERN = re.compile(r'\s+')

def normalize_label(value: str) -> int:
    lowered = value.strip().lower()
    if lowered in {'1', 'spam', 'yes', 'true', 'positive'}:
        return 1
    if lowered in {'0', 'ham', 'not_spam', 'not spam', 'no', 'false', 'negative'}:
        return 0
    raise ValueError(f'Label tidak dikenali: {value!r}')

def normalize_text(text: str) -> str:
    cleaned = text.lower().strip()
    cleaned = URL_PATTERN.sub(' URL ', cleaned)
    cleaned = EMAIL_PATTERN.sub(' EMAIL ', cleaned)
    cleaned = re.sub(r'[^\\w\\s@.+-]', ' ', cleaned)
    cleaned = WHITESPACE_PATTERN.sub(' ', cleaned)
    return cleaned

def detect_column(fieldnames, candidates):
    lookup = {name.strip().lower(): name for name in fieldnames}
    for candidate in candidates:
        if candidate in lookup:
            return lookup[candidate]
    raise ValueError(f'Kolom tidak ditemukan. Kandidat: {candidates}')

def is_count_dataset(fieldnames):
    lowered = {name.strip().lower() for name in fieldnames}
    return 'prediction' in lowered and any(name not in IGNORE_COLUMNS for name in lowered)

@dataclass
class TextDataset:
    texts: list[str]
    labels: list[int]

@dataclass
class FeatureDataset:
    features: list[list[float]]
    labels: list[int]
    feature_names: list[str]

def load_text_dataset(csv_path: Path) -> TextDataset:
    with csv_path.open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        if not reader.fieldnames:
            raise ValueError('CSV tidak memiliki header kolom.')

        text_key = detect_column(reader.fieldnames, TEXT_COLUMNS)
        label_key = detect_column(reader.fieldnames, LABEL_COLUMNS)

        texts: list[str] = []
        labels: list[int] = []
        for row in reader:
            text_value = normalize_text((row.get(text_key) or '').strip())
            label_value = (row.get(label_key) or '').strip()
            if not text_value or not label_value:
                continue
            try:
                labels.append(normalize_label(label_value))
            except ValueError:
                continue
            texts.append(text_value)

    if not texts:
        raise ValueError('Dataset teks kosong atau tidak valid.')
    return TextDataset(texts=texts, labels=labels)

def load_feature_dataset(csv_path: Path) -> FeatureDataset:
    with csv_path.open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        if not reader.fieldnames:
            raise ValueError('CSV tidak memiliki header kolom.')

        label_key = detect_column(reader.fieldnames, ('prediction',))
        feature_names = [name for name in reader.fieldnames if name.strip().lower() not in IGNORE_COLUMNS]

        features: list[list[float]] = []
        labels: list[int] = []
        for row in reader:
            label_value = (row.get(label_key) or '').strip()
            if not label_value:
                continue
            try:
                labels.append(normalize_label(label_value))
            except ValueError:
                continue

            row_features: list[float] = []
            for name in feature_names:
                raw_value = (row.get(name) or '0').strip()
                try:
                    row_features.append(float(raw_value))
                except ValueError:
                    row_features.append(0.0)
            features.append(row_features)

    if not features:
        raise ValueError('Dataset fitur kosong atau label tidak valid.')
    return FeatureDataset(features=features, labels=labels, feature_names=feature_names)

def build_text_model() -> Pipeline:
    return Pipeline([
        ('features', FeatureUnion([
            ('word_tfidf', TfidfVectorizer(lowercase=False, ngram_range=(1, 2), sublinear_tf=True)),
            ('char_tfidf', TfidfVectorizer(lowercase=False, analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True)),
        ])),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ])

def build_feature_model() -> MultinomialNB:
    return MultinomialNB()

In [22]:
def normalize_text(text: str) -> str:
    cleaned = text.lower().strip()
    cleaned = URL_PATTERN.sub(' URL ', cleaned)
    cleaned = EMAIL_PATTERN.sub(' EMAIL ', cleaned)
    cleaned = re.sub(r'[^\w\s@.+-]', ' ', cleaned)
    cleaned = WHITESPACE_PATTERN.sub(' ', cleaned)
    return cleaned

In [18]:
def build_text_model() -> Pipeline:
    return Pipeline([
        ('features', FeatureUnion([
            ('word_tfidf', TfidfVectorizer(lowercase=False, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
            ('char_tfidf', TfidfVectorizer(lowercase=False, analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
        ])),
        ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced', C=4.0, solver='liblinear')),
    ])

In [23]:
print('Dataset ada:', DATASET_PATH.exists())
if not DATASET_PATH.exists():
    raise FileNotFoundError(DATASET_PATH)

with DATASET_PATH.open('r', encoding='utf-8-sig', newline='') as handle:
    sample_reader = csv.DictReader(handle)
    fieldnames = sample_reader.fieldnames or []

print('Kolom dataset:')
print(fieldnames[:15], '... total =', len(fieldnames))

if is_count_dataset(fieldnames):
    dataset = load_feature_dataset(DATASET_PATH)
    X = dataset.features
    y = dataset.labels
    model = build_feature_model()
    dataset_kind = 'feature_count'
    print('Terdeteksi dataset fitur hitungan kata dengan label Prediction.')
else:
    dataset = load_text_dataset(DATASET_PATH)
    X = dataset.texts
    y = dataset.labels
    model = build_text_model()
    dataset_kind = 'text'
    print('Terdeteksi dataset teks mentah.')

print('Jumlah sampel:', len(X))
print('Distribusi label:', {0: y.count(0), 1: y.count(1)})

Dataset ada: True
Kolom dataset:
['Category', 'Message'] ... total = 2
Terdeteksi dataset teks mentah.
Jumlah sampel: 5572
Distribusi label: {0: 4825, 1: 747}


In [24]:
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if len(set(y)) > 1 else None,
)

model.fit(x_train, y_train)
predictions = model.predict(x_test)

accuracy = accuracy_score(y_test, predictions)
conf_matrix = confusion_matrix(y_test, predictions)
report = classification_report(y_test, predictions, target_names=['ham', 'spam'], zero_division=0)

print('Accuracy:', round(accuracy, 4))
print('Confusion matrix:\n', conf_matrix)
print('Classification report:\n', report)

Accuracy: 0.991
Confusion matrix:
 [[964   2]
 [  8 141]]
Classification report:
               precision    recall  f1-score   support

         ham       0.99      1.00      0.99       966
        spam       0.99      0.95      0.97       149

    accuracy                           0.99      1115
   macro avg       0.99      0.97      0.98      1115
weighted avg       0.99      0.99      0.99      1115



In [25]:
joblib.dump(model, MODEL_PATH)
print('Model disimpan ke:', MODEL_PATH.resolve())

Model disimpan ke: C:\MLLECSESSION11\spam_model.joblib


In [26]:
sample_inputs = [
    normalize_text('You won a big prize, click this link now'),
    normalize_text('Hello, please review the meeting document tomorrow morning'),
]

if dataset_kind == 'feature_count':
    print('Notebook ini memakai model fitur hitungan kata. Untuk prediksi manual, gunakan model teks mentah atau siapkan vektor fitur yang sesuai dengan kolom dataset.')
else:
    probs = model.predict_proba(sample_inputs)
    preds = model.predict(sample_inputs)
    for text, pred, prob in zip(sample_inputs, preds, probs):
        print({
            'text': text,
            'prediction': 'spam' if int(pred) == 1 else 'ham',
            'spam_probability': float(prob[1]),
        })

{'text': 'you won a big prize click this link now', 'prediction': 'spam', 'spam_probability': 0.7813637495977781}
{'text': 'hello please review the meeting document tomorrow morning', 'prediction': 'ham', 'spam_probability': 0.0572793663527518}


In [27]:
from pathlib import Path
import json

metadata_path = Path('spam_model_metadata.json')
metadata = {
    'dataset_kind': dataset_kind,
    'model_path': str(MODEL_PATH),
}
if dataset_kind == 'feature_count':
    metadata['feature_names'] = dataset.feature_names
else:
    metadata['text_columns'] = list(TEXT_COLUMNS)
    metadata['label_columns'] = list(LABEL_COLUMNS)
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Metadata disimpan ke:', metadata_path.resolve())


Metadata disimpan ke: C:\MLLECSESSION11\spam_model_metadata.json
